# Занятие 8, пара 2. Разведочный анализ и дашборд


Автор ноутбука - Дуркин Анатолий Альбертович

Старший преподаватель кафедры прикладной математики и компьютерных наук СГУ им. Питирима Сорокина

Замечания, предложения, идеи, вопросы, связь с автором:
- anatoliy.durkin@mail.ru
- Telegram - @AnatoDu

Больше информации и материалов на канале автора: https://t.me/code_matan_ai


**Как проходить тетрадь.** Основные примеры работают без ответов «Ваш ход».
        Если пропускаете упражнение, пропустите также его `check(...)`. Продолжайте следующий обычный раздел.
        Сохраняйте свои попытки в отдельной копии. Проверки помогают разобраться, но не выставляют баллы.
        Интернет, Git и регистрация для основной работы не нужны.


## От картинки к проверяемому выводу

В 08.1 мы выбирали график под вопрос. Теперь соберём маленькое исследование, которое другой человек
сможет проверить: **вопрос → нужные строки → расчёт → график → вывод с ограничением**.
После пары останутся таблица результатов, PNG и автономный HTML-отчёт с выбором острова.
Для основной работы достаточно Pandas и Matplotlib. Данные те же, но всё загружается заново.
В конце — переход к ЛР4 и хранилищу DuckDB из блока 07; установка на паре не требуется.


### Подключим учебные файлы

        Следующая ячейка — готовая служебная настройка. Она ищет `course_support` в рабочей папке
        или выше. Её не надо переписывать по памяти. `DATA_DIR` указывает на учебные данные,
        `WORKSPACE_DIR` — на корень вашей копии, `check` запускает проверку упражнения.
        Если поддержка не найдена, перенесите весь комплект, а не только notebook.
        В этой тетради создаются все необходимые переменные: другую тетрадь запускать не нужно.


In [ ]:
import sys
from pathlib import Path

# Одинаковый запуск в репозитории курса и в личном проекте.
for WORKSPACE_DIR in (Path.cwd(), *Path.cwd().parents):
    if (WORKSPACE_DIR / 'course_support' / '__init__.py').is_file():
        break
else:
    raise FileNotFoundError('Откройте тетрадь внутри папки с course_support; нужен весь комплект проекта')

# При переключении курса в том же kernel не оставляем чужую поддержку в памяти.
loaded = sys.modules.get('course_support')
expected = WORKSPACE_DIR / 'course_support' / '__init__.py'
if loaded is not None and Path(loaded.__file__).resolve() != expected.resolve():
    for name in list(sys.modules):
        if name == 'course_support' or name.startswith('course_support.'):
            del sys.modules[name]
sys.path.insert(0, str(WORKSPACE_DIR))
from course_support import SUPPORT_DIR, DATA_DIR
COURSE_DIR = SUPPORT_DIR
# Только упражнения о личном README: в репозитории курса это ещё чистый шаблон.
PROJECT_DIR = WORKSPACE_DIR / 'project_template' if (WORKSPACE_DIR / 'project_template').is_dir() else WORKSPACE_DIR
from course_support.checks import check

PROJECT_DIR


### 1. Не будем начинать с «нарисуем всё»

Разведочный анализ (EDA) помогает разобраться, что есть в данных, найти закономерности и проверить
первые предположения. Если построить пятьдесят графиков, что-нибудь интересное найдётся почти наверняка.
Поэтому начнём с трёх вопросов:

1. Сколько наблюдений можно использовать для анализа массы?
2. Как различается масса между видами **в этой выборке**?
3. Можно ли объяснить различие островов самим островом, если виды представлены на них по-разному?

Источники набора и единицы описаны в 08.1 и `data/visualization/SOURCE.md` внутри course_support.
`isna().sum()` показывает пропуски по столбцам; для каждого вопроса выбираем свой необходимый набор полей.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

penguins = pd.read_csv(DATA_DIR / 'visualization' / 'penguins.csv')
print('Всего строк:', len(penguins))
print(penguins.isna().sum())
known_mass = penguins.dropna(subset=['body_mass_g'])
print('С известной массой:', len(known_mass))


### 2. Сначала числа, затем оформление

В блоке 06 мы давали агрегатам имена через `agg`. Повторим на одном показателе.
`count` считает известные значения выбранного столбца; `mean` — среднее, `median` — середину
упорядоченного списка. При чётном количестве медиана — среднее двух центральных значений.
Покажем и число измерений, и среднюю массу: среднее без n скрывает размер группы.


In [ ]:
by_species = penguins.groupby('species').agg(
    n=('body_mass_g', 'count'),
    mean_g=('body_mass_g', 'mean'),
    median_g=('body_mass_g', 'median'),
)
display(by_species.round(1))


Теперь средняя масса — высота столбца, а вид — категория. Это уже не количество птиц из 08.1.
`round(1)` выше округлял отображаемую таблицу; расчёты продолжаем на неокруглённых данных.
Среднее удобно сравнивать, но оно не показывает разброс отдельных измерений.


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(by_species.index, by_species['mean_g'])
ax.set_ylabel('Средняя масса, г')
ax.set_xlabel('Вид')
ax.set_title('Средняя масса среди известных измерений')
plt.show()


Проверяемый вывод: «В этом наборе у Gentoo средняя масса больше, чем у двух других видов».
Непроверяемая добавка: «поэтому именно этот вид лучше приспособлен к жизни». Второй части нет в измерениях.
Чтобы не прятать разброс за средними, в отчёте ниже оставим точки каждой птицы.


### ✏️ Ваш ход — Удалить лишнее — не всегда очистить

Коллега начал анализ массы с `penguins.dropna()` без subset.
Посчитайте `lost_for_other_reason`: сколько пригодных **для анализа массы** строк он лишился дополнительно.
Не записывайте ответ по увиденному числу: получите его из таблицы.
Укажите, для какого другого вопроса более строгое удаление могло бы быть оправдано.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.1', lost_for_other_reason)


**Мой вывод:** …


### 3. Срез: тот же вопрос на части данных

Посмотрим средние массы по островам. Можно ли теперь сказать, что остров «делает птиц тяжелее»?
Прежде чем отвечать, проверим **состав групп**. Срез — отбор строк по условию, а не новый набор,
который можно трактовать как независимый эксперимент.


In [ ]:
by_island = penguins.groupby('island').agg(
    n=('body_mass_g', 'count'),
    mean_g=('body_mass_g', 'mean'),
)
display(by_island.round(1))
composition = penguins.groupby(['island', 'species']).size()
print(composition)


Острова представлены разными наборами видов. В частности, Gentoo в этой таблице наблюдали на Biscoe,
а Chinstrap — на Dream. Если смешать виды, средняя масса острова будет зависеть и от их долей.
Сравним **один вид** между островами. `loc` с булевым условием мы уже использовали при очистке.


In [ ]:
adelie = penguins.loc[penguins['species'] == 'Adelie']
adelie_islands = adelie.groupby('island').agg(
    n=('body_mass_g', 'count'),
    mean_g=('body_mass_g', 'mean'),
)
display(adelie_islands.round(1))


Сравнение внутри вида отвечает более узкому вопросу. Оно всё ещё не превращается в причинный
эксперимент: могут различаться пол, возраст, время и способ отбора. Хороший отчёт даёт проверить
конкретный вывод, а не обещает объяснить всё устройство Антарктиды.

### 4. Где у среднего спрятались веса

Если известны средние двух групп и их размеры, общее среднее равно сумме
`среднее × размер`, делённой на общее число наблюдений. Мы встречали эту ловушку в хранилищах;
в дашбордах она появляется снова, когда крупные карточки сводят в одну.


In [ ]:
group_means = [10, 20]
group_sizes = [2, 8]
total = group_means[0] * group_sizes[0] + group_means[1] * group_sizes[1]
weighted_mean = total / sum(group_sizes)
print('Среднее всех наблюдений:', weighted_mean)
print('Просто среднее двух средних:', sum(group_means) / 2)


### ✏️ Ваш ход — Поймать ошибку в отчёте

В вымышленной службе поддержки в понедельник обработали 2 обращения со средним временем 10 минут,
во вторник — 8 обращений со средним временем 20 минут. На карточке «среднее за два дня» написано 15.

Напишите функцию `combine_means(means, sizes)`, которая принимает списки средних и численностей
и возвращает среднее **по обращениям**. Если суммарно обращений нет, верните `None`.
Цикл по `zip(means, sizes)` позволяет получить пару значений на каждом шаге: среднее из первого списка и соответствующий размер из второго.
Объясните, на какой вопрос число 15 всё-таки отвечает.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.2', combine_means)


**Мой вывод:** …


### 5. Отдельный рисунок для одного среза

Сделаем отчёт, в котором можно выбрать остров. Сначала рисуем **один** срез обычным кодом.
По горизонтали — вид, по вертикали — масса каждой птицы; поэтому используем scatter, а не столбец среднего.
`plt` умеет размещать строковые категории на оси. Точки одного вида стоят в одном столбце,
прозрачность частично помогает увидеть наложения. Точное количество показываем отдельно.

`set_ylim(2500, 6500)` фиксирует одинаковый масштаб для всех будущих срезов.
Иначе переключение фильтра могло бы визуально менять разброс только из-за масштаба.


In [ ]:
island_part = known_mass.loc[known_mass['island'] == 'Biscoe']
fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(island_part['species'], island_part['body_mass_g'], alpha=0.5)
ax.set_ylim(2500, 6500)
ax.set_xlabel('Вид')
ax.set_ylabel('Масса, г')
ax.set_title('Biscoe: отдельные измерения')
plt.show()


Теперь повторим те же команды для любого среза и сохраним рисунок. Функция принимает таблицу,
название и путь. `fig.savefig` разобрали в 08.1; `plt.close(fig)` закрывает готовую фигуру,
чтобы при сборке нескольких срезов Jupyter не показывал их все подряд.
Для пустого среза `ax.text` пишет сообщение в точке (0.5, 0.5) **относительно области графика**:
это задаёт `transform=ax.transAxes`. Пустота не должна превращаться в выдуманный ноль.


In [ ]:
def save_slice_plot(part, title, path):
    fig, ax = plt.subplots(figsize=(8, 4))
    if len(part) == 0:
        ax.text(0.5, 0.5, 'Нет измерений', transform=ax.transAxes, ha='center')
    else:
        ax.scatter(part['species'], part['body_mass_g'], alpha=0.5)
    ax.set_ylim(2500, 6500)
    ax.set_xlabel('Вид')
    ax.set_ylabel('Масса, г')
    ax.set_title(title)
    fig.tight_layout()
    fig.savefig(path, dpi=120)
    plt.close(fig)

output_dir = WORKSPACE_DIR / 'outputs' / 'penguins_report'
output_dir.mkdir(parents=True, exist_ok=True)
save_slice_plot(island_part, 'Biscoe', output_dir / 'biscoe.png')


### 6. Соберём срезы и упакуем отчёт

Словарь `slices` связывает название фильтра с таблицей. `items()` даёт название и таблицу в цикле.
`enumerate(...)` добавляет порядковый номер; здесь он нужен только для имён PNG.
Список `views` содержит уже рассчитанные карточки. Поле `count` — число строк **с известной массой**,
`mean` — их средняя масса. Для пустой таблицы среднее неизвестно (`None`), а не равно нулю.


In [ ]:
slices = {'Все острова': known_mass}
for island, part in known_mass.groupby('island'):
    slices[island] = part

views = []
for number, (label, part) in enumerate(slices.items()):
    image_path = output_dir / f'slice_{number}.png'
    save_slice_plot(part, label, image_path)
    mean_mass = None
    if len(part) > 0:
        mean_mass = float(part['body_mass_g'].mean())
    views.append({
        'label': label,
        'count': len(part),
        'mean': mean_mass,
        'note': 'Сравните состав видов. Средняя по острову смешивает разные группы.',
        'image': image_path,
    })
print([(view['label'], view['count']) for view in views])


`export_report` — готовая **служебная упаковка** из course_support, а не новый способ анализа.
Она принимает список карточек и путь, встраивает PNG в HTML и добавляет выбор среза в браузере.
Расчёты выше сделали мы; JavaScript интерфейса в этом курсе писать не требуется.
Файл работает без сервера и интернета. Его можно открыть двойным щелчком в файловом менеджере
или скачать из Jupyter и открыть браузером. При изменении данных нужно пересчитать ячейки и экспортировать заново.


In [ ]:
from course_support.report import export_report

report_path = export_report(views, output_dir / 'report.html')
by_species.to_csv(output_dir / 'by_species.csv')
print('Откройте в браузере:', report_path)


**Проверка руками:** выберите Biscoe, затем Torgersen. Изменились n, средняя масса и набор видов?
Найдите вывод, который можно проверить этим фильтром, и вывод, который нельзя.
Это минимальный интерактивный отчёт. Кнопки сами по себе не делают вывод качественным.
Для ЛР4 можно использовать другой инструмент, если он доступен и даёт полезный срез.


### ✏️ Ваш ход — Проверить, что пустота не притворяется данными

Напишите `mass_card(frame)`: функция возвращает словарь `{'n': ..., 'mean_g': ...}`
для известной массы в переданной таблице. Если известных измерений нет, среднее — `None`.
Проверьте на всех птицах, пустой таблице и строках только с пропусками массы.
После проверки откройте свой отчёт и опишите один сценарий, которым вы бы проверили фильтр.
Не используйте количество всех строк как знаменатель среднего известной массы.


In [ ]:
# ✏️ ваш код здесь


In [ ]:
check('08.2.3', mass_card)


**Мой вывод:** …


## ЛР4 — связать хранилище и вывод

Задание и прежние критерии на 20 баллов — [ЛР4](../docs/ЛР4.md) в шаблоне проекта;
при работе из корня откройте `docs/ЛР4.md`. Начните с одного вопроса к своему проекту,
одной витрины и одного полезного фильтра. Три десятка графиков не требуются.

**Важное разделение:** сегодня анализ работает без установки новой библиотеки. В ЛР4 остаётся
DuckDB-хранилище из блока 07: raw, clean, views, ключи/связи, GROUP BY, JOIN и окно.
Если JOIN или окно не нужны предметной задаче, покажите навык на учебном наборе 06–07,
а не придумывайте бессмысленную операцию ради галочки. Сдача файлами и архивом равноправна GitHub.

### Дополнительный мост к DuckDB — 10–15 минут

Сначала проверьте, доступен ли DuckDB. При отсутствии пакет можно установить позже в своё окружение;
основной отчёт выше уже работает. Не запускаем установку автоматически на чужом компьютере.
Далее — команды, знакомые по 07: открыть базу, зарегистрировать DataFrame как входную таблицу,
создать слои и представление, выполнить запрос и закрыть соединение.
`CREATE OR REPLACE` пересобирает только таблицы учебной базы по отдельному пути.


In [ ]:
try:
    import duckdb
except ImportError:
    duckdb = None
    print('DuckDB недоступен. Основной отчёт готов; этот мост выполните позже.')


Здесь `raw_penguins` — исходная **копия опубликованного CSV**, а не все полевые записи исследователей.
`clean_penguins` отбирает строки для анализа массы; для другого вопроса правило может отличаться.
Номер строки — технический ключ только этого снимка, он не заменяет биологический ID птицы.
VIEW хранит SQL определения витрины. Отчёт должен читать результат её запроса, чтобы не считать
один показатель разными правилами в двух местах. Эта демонстрация сама по себе не закрывает всю ЛР4.


In [ ]:
if duckdb is not None:
    raw_input = penguins.copy()
    raw_input['row_id'] = range(1, len(raw_input) + 1)
    database_path = output_dir / 'penguins_practice.duckdb'
    connection = duckdb.connect(str(database_path))
    connection.register('csv_input', raw_input)
    connection.execute('CREATE OR REPLACE TABLE raw_penguins AS SELECT * FROM csv_input')
    connection.execute("""
        CREATE OR REPLACE TABLE clean_penguins AS
        SELECT * FROM raw_penguins WHERE body_mass_g IS NOT NULL
    """)
    connection.execute("""
        CREATE OR REPLACE VIEW species_mass AS
        SELECT species, COUNT(*) AS n, AVG(body_mass_g) AS mean_g
        FROM clean_penguins GROUP BY species
    """)
    sql_result = connection.execute('SELECT * FROM species_mass ORDER BY species').df()
    display(sql_result)
    sql_result.to_csv(output_dir / 'species_mass_from_sql.csv', index=False)
    connection.close()


## Что должно остаться после пары

Тетрадь с выводами и папка `outputs/penguins_report`: таблица, рисунки, `report.html`.
Для передачи только отчёта достаточно HTML: картинки уже внутри. Для воспроизведения нужны также тетрадь,
данные и зависимости. Не путайте «посмотреть результат» с «пересчитать результат».

**Резерв на 10–15 минут:** добавьте полезный срез по виду вместо острова. Сначала сформулируйте,
какой вывод он проверяет, затем получите таблицы. Если сделаете оба фильтра, подумайте, что увидит
человек при выборе вида и острова, для которых в наборе нет наблюдений.

По желанию: Коул Нафлик, «Сторителлинг на языке данных». Для техники —
[Matplotlib: интерфейс Figure/Axes](https://matplotlib.org/stable/users/explain/quick_start.html).
Для следующего блока сохраняем привычку документировать решения: скоро будем обсуждать разметку и качество данных.
